In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, auc, average_precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, precision_score, hamming_loss
from sklearn.metrics import brier_score_loss


from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

import copy
import torch
import torch.nn as nn
from blitz.modules import BayesianLinear

In [2]:
import pickle
with open('balanced_data.pkl', 'rb') as b:
    data = pickle.load(b)

X_res_sk = data['X_res_stroke']
y_res_sk = data['y_res_stroke']
X_res_db = data['X_res_diab']
y_res_db = data['y_res_diab']
X_res_cn = data['X_res_can']
y_res_cn = data['y_res_can']       
X_res_ha = data['X_res_ha']
y_res_ha = data['y_res_ha']    

X_val = data['X_val'] 
y_val = data['y_val']   

X_test = data['X_test']
y_test = data['y_test']

X_bal=data['X_bal']
y_bal=data['y_bal']

y_test=y_test.rename(columns=
              {'heart_attack' : 'HA'})
y_bal=y_bal.rename(columns=
              {'heart_attack' : 'HA'})
y_val=y_val.rename(columns=
              {'heart_attack' : 'HA'})

In [ ]:
scaler = MinMaxScaler() 
X_train_st = scaler.fit_transform(X_bal)    
X_val_st   = scaler.transform(X_val)        
X_test_st = scaler.transform(X_test)       

x_train_t = torch.tensor(X_train_st, dtype=torch.float32)
y_train_t = torch.tensor(y_bal.values, dtype=torch.float32)

x_val_t   = torch.tensor(X_val_st, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

x_test_t  = torch.tensor(X_test_st, dtype=torch.float32)

In [4]:
def youden_threshold(y_true_col, y_prob_col):
    fpr, tpr, thresholds = roc_curve(y_true_col, y_prob_col)
    return thresholds[np.argmax(tpr - fpr)]

pos_weight = torch.tensor([
    (y_bal[col] == 0).sum() / (y_bal[col] == 1).sum()
    for col in y_bal.columns], dtype=torch.float32)

***BAYESIAN NEURAL NETWORK***

Modify to be comparable to MLP 

In [ ]:
class BNN(nn.Module):
    def __init__(self,n_features,n_outputs,M):
        super().__init__()
        self.fc1 = BayesianLinear(n_features,M)
        self.fc2 = BayesianLinear(M,M)
        self.fc3 = BayesianLinear(M,n_outputs)
        self.relu = nn.ReLU()
    

    def forward(self,x):
        x = self.relu(self.fc1(x)) 
        x = self.relu(self.fc2(x))
        return self.fc3(x)
    
    def kl_loss(self):
        kl = 0
        for layer in [self.fc1, self.fc2, self.fc3]:
            mu = layer.weight_sampler.mu
            rho = layer.weight_sampler.rho
            sigma = torch.log1p(torch.exp(rho))
            kl_w = 0.5*(mu**2 + sigma**2 - torch.log(sigma**2)-1).sum()
            
            mu_b = layer.bias_sampler.mu
            rho_b = layer.bias_sampler.rho
            sigma_b = torch.log1p(torch.exp(rho_b))  # = log(1+exp(rho))
            kl_b = 0.5*(mu_b**2 + sigma_b**2 - torch.log(sigma_b**2)-1).sum()  
            
            
            kl += kl_w + kl_b
        return kl 

In [ ]:
torch.manual_seed(1234)

n_features  = X_train_st.shape[1]
n_outputs   = y_bal.shape[1]
n_epochs   = 1000
n_restarts = 5
N_train    = x_train_t.shape[0]
loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
n_sample   = 30

M_val      = [16, 32, 64]
kl_weights = [10/N_train, 1/N_train, 1/(10*N_train), 1/(100*N_train)]

tuning_res = []

for kl_w in kl_weights:
    for M in M_val:

        best_misc  = np.inf
        best_model = None

        for h in range(n_restarts):
            model = BNN(n_features, n_outputs, M)
            opt   = torch.optim.Adam(model.parameters(), lr=0.001)

            for epoch in range(n_epochs):
                model.train()
                opt.zero_grad()
                loss = loss_fn(model(x_train_t), y_train_t) + kl_w * model.kl_loss()
                loss.backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                y_prob_val_h = torch.stack([
                    torch.sigmoid(model(x_val_t))
                    for _ in range(n_sample)
                ]).mean(0).numpy()

            errors_h = []
            for i, col in enumerate(y_val.columns):
                thr      = youden_threshold(y_val[col].values, y_prob_val_h[:, i])
                y_pred_i = (y_prob_val_h[:, i] >= thr).astype(int)
                errors_h.append((y_pred_i != y_val[col].values).mean())
            misc_h = np.mean(errors_h)

            if misc_h < best_misc:
                best_misc  = misc_h
                best_model = copy.deepcopy(model)

        best_model.eval()
        with torch.no_grad():
            y_prob_val = torch.stack([
                torch.sigmoid(best_model(x_val_t))
                for _ in range(n_sample)
            ]).mean(0).numpy()

        errors = []
        for i, col in enumerate(y_val.columns):
            thr      = youden_threshold(y_val[col].values, y_prob_val[:, i])
            y_pred_i = (y_prob_val[:, i] >= thr).astype(int)
            errors.append((y_pred_i != y_val[col].values).mean())
        misc = np.mean(errors)

        print(f"kl_w={kl_w:.2e}  M={M}  misc={misc:.4f}")
        tuning_res.append((kl_w, M, misc))

best_kl_w, best_M, best_misclass = min(tuning_res, key=lambda x: x[2])
print(f"\nMiglior KL_w={best_kl_w:.2e}, M={best_M}, Misclassification={best_misclass:.4f}")

In [7]:
best_misc_final = np.inf
model_final     = None
torch.manual_seed(1234)

for h in range(n_restarts):
    model = BNN(n_features, n_outputs, best_M)
    opt   = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(n_epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(x_train_t), y_train_t) + best_kl_w * model.kl_loss()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        y_prob_val_h = torch.stack([
            torch.sigmoid(model(x_val_t))
            for _ in range(n_sample)
        ]).mean(0).numpy()

    errors_h = []
    for i, col in enumerate(y_val.columns):
        thr      = youden_threshold(y_val[col].values, y_prob_val_h[:, i])
        y_pred_i = (y_prob_val_h[:, i] >= thr).astype(int)
        errors_h.append((y_pred_i != y_val[col].values).mean())
    misc_h = np.mean(errors_h)

    if misc_h < best_misc_final:
        best_misc_final = misc_h
        model_final     = copy.deepcopy(model)

print(f'Best misc finale: {best_misc_final:.4f}')

Best misc finale: 0.3480


In [ ]:

n_sample=100
model_final.eval()

with torch.no_grad():
    pred_val = torch.stack([ 
        torch.sigmoid(model_final(x_val_t))
        for _ in range(n_sample)]) 

y_prob_val = pred_val.mean(0).numpy()


best_thresholds_bnn = {}
for i, col in enumerate(y_val.columns):
    fpr, tpr, thresholds = roc_curve(y_val[col], y_prob_val[:, i])
    best_t = thresholds[np.argmax(tpr - fpr)]
    best_thresholds_bnn[col] = best_t
    print(f"{col}: threshold = {best_t:.3f}")
    

model_final.eval()
with torch.no_grad():
    pred = torch.stack([ 
        torch.sigmoid(model_final(x_test_t))
        for _ in range(n_sample)
    ]) 
y_prob_test = pred.mean(0).numpy()   
y_std_test  = pred.std(0).numpy()


results_bnn = {}
for i, col in enumerate(y_test.columns):
    y_pred_i = (y_prob_test[:,i] >= best_thresholds_bnn[col]).astype(int)
    
    precision, recall, _ = precision_recall_curve(y_test[col].values, y_prob_test[:,i])
    
    # Ambiguity (predictive entropy)
    p = y_prob_test[:, i]
    eps = 1e-8
    entropy_i = - p * np.log2(p + eps) - (1 - p) * np.log2(1 - p + eps)
    
    results_bnn[col] = {
        'f1' : f1_score(y_test[col],y_pred_i),
        'precision' : precision_score(y_test[col],y_pred_i),
        'auc' : roc_auc_score(y_test[col], y_prob_test[:,i]),
        'auc-pr' : auc(recall, precision),
        'sens' : recall_score(y_test[col].values, y_pred_i),
        'uncertainty' : y_std_test[:,i].mean()*100, 
        'ambiguity' : entropy_i.mean(),
        'confusion' : confusion_matrix(y_test[col],y_pred_i),
        'misc' : (y_pred_i != y_test[col].values).mean(),
        "brier": brier_score_loss(y_test[col].values, y_prob_test[:, i]),
        'threshold': best_thresholds_bnn[col]
    }
    
y_pred_multi = np.zeros_like(y_test.values) 
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_prob_test[:, i] >= best_thresholds_bnn[col]).astype(int)

macro_f1 = f1_score(y_test.values, y_pred_multi, average='macro')
hamming = hamming_loss(y_test.values, y_pred_multi)

results_bnn["MacroF1"] = macro_f1
results_bnn["Hamming"] = hamming

brier_mean = np.mean([
    brier_score_loss(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])
results_bnn["Brier_mean"] = brier_mean

macro_auc = np.mean([
    roc_auc_score(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])
macro_auc_pr = np.mean([
    average_precision_score(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])

results_bnn["MacroAUC"] = macro_auc
results_bnn["MacroAUC_PR"] = macro_auc_pr

print(pd.DataFrame({k: v for k, v in results_bnn.items() if k not in ['confusion','threshold']}).T)

diabetes: threshold = 0.491
stroke: threshold = 0.262
HA: threshold = 0.413
cancer: threshold = 0.249
                   f1 precision       auc    auc-pr      sens uncertainty  \
diabetes     0.454902  0.318876  0.831343   0.45407  0.793313    0.794787   
stroke        0.10999  0.059628   0.72332  0.084335  0.707792    1.120675   
HA           0.319018   0.19883  0.732035  0.237979  0.806569    0.705726   
cancer       0.103064  0.056719  0.606543  0.060515  0.563452    1.251685   
MacroF1      0.246744  0.246744  0.246744  0.246744  0.246744    0.246744   
Hamming      0.370527  0.370527  0.370527  0.370527  0.370527    0.370527   
Brier_mean   0.145612  0.145612  0.145612  0.145612  0.145612    0.145612   
MacroAUC      0.72331   0.72331   0.72331   0.72331   0.72331     0.72331   
MacroAUC_PR  0.210915  0.210915  0.210915  0.210915  0.210915    0.210915   

            ambiguity                   confusion      misc     brier  \
diabetes     0.733551  [[2838, 1115], [136, 522]]  0.2

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    quantiles = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    ece = 0
    n = len(y_prob)
    for b in range(n_bins):
        mask = (y_prob >= quantiles[b]) & (y_prob < quantiles[b+1])
        if mask.sum() > 0:
            acc = y_true[mask].mean()   
            conf = y_prob[mask].mean()  
            ece += mask.mean() * abs(acc - conf)
    return ece


for i, col in enumerate(y_test.columns):
    ece = expected_calibration_error(
        y_test[col].values, 
        y_prob_test[:, i]
    )
    print(f'{col}: ECE = {ece:.4f}')
    results_bnn[col]["ece"] = ece


diabetes: ECE = 0.2441
stroke: ECE = 0.2140
HA: ECE = 0.2954
cancer: ECE = 0.2080


In [ ]:
for i, col in enumerate(y_test.columns):
    p = y_prob_test[:, i]
    eps = 1e-8
    entropy_i = - p * np.log2(p + eps) - (1 - p) * np.log(1 - p + eps)
    
    mask_pos = y_test[col].values == 1
    mask_neg = y_test[col].values == 0
    
    print(f"\n{col}:")
    print(f"  ambiguity positivi : {entropy_i[mask_pos].mean():.4f}")
    print(f"  ambiguity negativi : {entropy_i[mask_neg].mean():.4f}")
    print(f"  mean_prob positivi : {p[mask_pos].mean():.4f}")
    print(f"  mean_prob negativi : {p[mask_neg].mean():.4f}")


diabetes:
  ambiguity positivi : 0.6421
  ambiguity negativi : 0.6269
  mean_prob positivi : 0.6598
  mean_prob negativi : 0.3416

stroke:
  ambiguity positivi : 0.7207
  ambiguity negativi : 0.5888
  mean_prob positivi : 0.3958
  mean_prob negativi : 0.2424

HA:
  ambiguity positivi : 0.7313
  ambiguity negativi : 0.6902
  mean_prob positivi : 0.5775
  mean_prob negativi : 0.3924

cancer:
  ambiguity positivi : 0.6924
  ambiguity negativi : 0.6300
  mean_prob positivi : 0.3062
  mean_prob negativi : 0.2484


In [12]:
with torch.no_grad():
    prova = torch.stack([
        model_final(x_test_t)
        for _ in range(n_sample)]) 

prova_std = prova.std(0).numpy()
print(prova_std)

[[0.04375038 0.08914117 0.03161273 0.08807747]
 [0.04569209 0.0629278  0.02794261 0.07256086]
 [0.04192759 0.06989147 0.03321705 0.08330715]
 ...
 [0.04149912 0.06133891 0.02746559 0.06832594]
 [0.04533085 0.07903189 0.02973001 0.0854413 ]
 [0.04732171 0.07215925 0.03224104 0.07943925]]


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_final = model_final.to(device)

def predict_single_patient_bnn(model, x_row, thresholds, n_samples=100):
    """
    - prob_mean: sigmoid
    - prob_std: std MC, uncertainty
    - prediction: classes 0/1 considering Youden Index
    """

    # Correct shape (1, n_features)
    x_np = np.array(x_row).reshape(1, -1)
    x_t  = torch.tensor(x_np, dtype=torch.float32).to(device)

    model.eval()
    samples = []

    with torch.no_grad():
        for _ in range(n_samples):
            out = model(x_t)
            prob = torch.sigmoid(out).cpu().numpy()
            samples.append(prob)

    samples = np.vstack(samples)    

    prob_mean = samples.mean(axis=0)
    prob_std  = samples.std(axis=0)
    
    eps = 1e-8
    ambiguity = - prob_mean * np.log2(prob_mean + eps) \
                - (1 - prob_mean) * np.log2(1 - prob_mean + eps)

    thr_vec = np.array([thresholds[c] for c in thresholds])
    prediction = (prob_mean >= thr_vec).astype(int)

    return prob_mean, prob_std, prediction, ambiguity



In [ ]:
patient_idx = 1810
x_one = X_test.iloc[patient_idx].values.reshape(1, -1)
x_one_std = scaler.transform(x_one)  

prob, unc, pred, ambiguity = predict_single_patient_bnn(
    model_final,
    x_one_std[0],
    best_thresholds_bnn,
    n_samples=100
)

df_patient = pd.DataFrame({
    "prob": prob,
    "uncertainty": unc *100,  # %
    'ambiguity' : ambiguity,
    "threshold": [best_thresholds_bnn[c] for c in y_test.columns],
    "prediction": pred,
    "True Pred": [y_test.iloc[patient_idx][c] for c in y_test.columns]
}, index=y_test.columns)

print(df_patient)


              prob  uncertainty  ambiguity  threshold  prediction  True Pred
diabetes  0.056865     0.291224   0.314872   0.490990           0          0
stroke    0.294629     1.648013   0.874626   0.261928           1          0
HA        0.297225     0.784826   0.877873   0.412565           0          0
cancer    0.670956     1.911801   0.913947   0.249187           1          0


/Users/samuelemazzei/Desktop/Python_tesi/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [ ]:
def get_single_patient_mc_samples(model, x_row, n_samples=500):
    x_np = np.array(x_row).reshape(1, -1)
    x_t = torch.tensor(x_np, dtype=torch.float32).to(device)

    model.eval()
    samples = []

    with torch.no_grad():
        for _ in range(n_samples):
            out = model(x_t)
            prob = torch.sigmoid(out).cpu().numpy().ravel()
            samples.append(prob)

    return np.vstack(samples)



x_one = X_test.iloc[patient_idx].values.reshape(1, -1)
x_one_std = scaler.transform(x_one)

patient_samples = get_single_patient_mc_samples(
    model_final,
    x_one_std[0],
    n_samples=500
)

patient_mean = patient_samples.mean(axis=0)
patient_std = patient_samples.std(axis=0)
patient_thresholds = np.array([best_thresholds_bnn[col] for col in y_test.columns])
patient_pred = (patient_mean >= patient_thresholds).astype(int)
patient_true = y_test.iloc[patient_idx].values

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

palette = sns.color_palette("Set2", n_colors=len(y_test.columns))

for i, (col, ax) in enumerate(zip(y_test.columns, axes)):
    sns.histplot(
        patient_samples[:, i],
        bins=25,
        stat="density",
        kde=True,
        color=palette[i],
        alpha=0.45,
        edgecolor="white",
        ax=ax
    )

    ax.axvline(
        patient_mean[i],
        color="#1f77b4",
        linewidth=2,
        label=f"mean = {patient_mean[i]:.3f}"
    )

    ax.axvline(
        patient_thresholds[i],
        color="#d62728",
        linestyle="--",
        linewidth=2,
        label=f"threshold = {patient_thresholds[i]:.3f}"
    )

    ax.set_title(
        f"{col} | true={patient_true[i]} | pred={patient_pred[i]} | std={patient_std[i]:.3f}",
        fontsize=11
    )
    ax.set_xlim(0, 1)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Density")
    ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
threshold_vec = np.array([best_thresholds_bnn[col] for col in y_test.columns])

y_pred_mc = (y_prob_test >= threshold_vec).astype(int)

distance_from_threshold = np.abs(y_prob_test - threshold_vec)

df_search = pd.DataFrame({
    "iloc_position": np.arange(len(y_test)),
    "uncertainty_mean": y_std_test.mean(axis=1),
    "distance_mean_threshold": distance_from_threshold.mean(axis=1),
    "n_correct": (y_pred_mc == y_test.values).sum(axis=1),
    "all_correct": (y_pred_mc == y_test.values).all(axis=1),
})

df_search["confidence_score"] = (
    df_search["distance_mean_threshold"] - df_search["uncertainty_mean"]
)

df_search.sort_values("confidence_score", ascending=False).head(20)

In [ ]:
n_samples_search = 300

model_final.eval()

with torch.no_grad():
    mc_probs_all = torch.stack([
        torch.sigmoid(model_final(x_test_t.to(device))).cpu()
        for _ in range(n_samples_search)
    ])

mc_probs_all = mc_probs_all.numpy()
# shape: (n_samples, n_patients, n_classes)

threshold_vec = np.array([best_thresholds_bnn[col] for col in y_test.columns])

mc_mean = mc_probs_all.mean(axis=0)
mc_std = mc_probs_all.std(axis=0)

# how often the MC samples exceed the threshold
cross_prob = (mc_probs_all >= threshold_vec).mean(axis=0)

rows = []

for patient_idx in range(len(y_test)):
    for class_idx, col in enumerate(y_test.columns):
        mean = mc_mean[patient_idx, class_idx]
        std = mc_std[patient_idx, class_idx]
        threshold = threshold_vec[class_idx]
        true = y_test.iloc[patient_idx, class_idx]
        pred = int(mean >= threshold)

        distance = abs(mean - threshold)

        # high score = more informative plot
        # close to the threshold + wide distribution + samples on both sides
        crossing_score = 1 - abs(cross_prob[patient_idx, class_idx] - 0.5) * 2

        interesting_score = (
            crossing_score * 2
            + std
            - distance
        )

        rows.append({
            "iloc_position": patient_idx,
            "class": col,
            "true": true,
            "pred": pred,
            "mean": mean,
            "threshold": threshold,
            "std": std,
            "distance_from_threshold": distance,
            "cross_prob": cross_prob[patient_idx, class_idx],
            "interesting_score": interesting_score
        })

df_interesting = pd.DataFrame(rows)

df_interesting.sort_values(
    "interesting_score",
    ascending=False
).head(20)

In [21]:
y_pred_mc = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_mc[:, i] = (y_prob_test[:, i] >= best_thresholds_bnn[col]).astype(int)

# Find indices where the model has at least one TP
tp_mask = ((y_pred_mc == 1) & (y_test.values == 1)).any(axis=1)
tp_indices = np.where(tp_mask)[0]
print(f"Pazienti con almeno un TP: {len(tp_indices)}")
print("Primi indici:", tp_indices[:10])

for idx in tp_indices[:20]:
    true = y_test.iloc[idx].values
    pred = y_pred_mc[idx]
    print(f"idx={idx} | True: {true} | Pred: {pred}")

Pazienti con almeno un TP: 975
Primi indici: [ 2 13 17 24 26 27 30 33 35 39]
idx=2 | True: [0 0 0 1] | Pred: [0 1 1 1]
idx=13 | True: [1 0 0 0] | Pred: [1 1 1 1]
idx=17 | True: [0 0 0 1] | Pred: [1 1 1 1]
idx=24 | True: [0 0 0 1] | Pred: [0 0 0 1]
idx=26 | True: [0 0 1 0] | Pred: [0 1 1 1]
idx=27 | True: [1 0 0 0] | Pred: [1 1 1 1]
idx=30 | True: [0 1 0 0] | Pred: [1 1 1 0]
idx=33 | True: [1 1 0 0] | Pred: [1 1 1 1]
idx=35 | True: [0 0 1 0] | Pred: [0 0 1 1]
idx=39 | True: [0 0 0 1] | Pred: [1 1 1 1]
idx=44 | True: [0 0 1 0] | Pred: [1 1 1 1]
idx=56 | True: [1 0 0 0] | Pred: [1 0 1 0]
idx=57 | True: [1 0 0 0] | Pred: [1 1 1 1]
idx=68 | True: [1 0 0 0] | Pred: [1 1 1 0]
idx=70 | True: [0 0 0 1] | Pred: [0 0 0 1]
idx=92 | True: [0 0 1 0] | Pred: [1 0 1 0]
idx=93 | True: [1 0 0 0] | Pred: [1 0 1 0]
idx=102 | True: [0 0 0 1] | Pred: [1 1 1 1]
idx=104 | True: [1 0 0 0] | Pred: [1 0 1 0]
idx=109 | True: [1 0 0 0] | Pred: [1 0 1 1]


In [ ]:
from pathlib import Path
import pickle

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "Models" else Path.cwd()

with open(PROJECT_DIR / "results_bnn.pkl", "wb") as f:
    pickle.dump(results_bnn, f)

print(f"Saved summary results to {PROJECT_DIR / 'results_bnn.pkl'}")


In [ ]:
from pathlib import Path
import pickle

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "Models" else Path.cwd()
patient_rows = []
eps = 1e-8

for i, col in enumerate(y_test.columns):
    p = y_prob_test[:, i]
    uncertainty = y_std_test[:, i]
    ambiguity = -p * np.log2(p + eps) - (1 - p) * np.log2(1 - p + eps)
    threshold = best_thresholds_bnn[col]
    prediction = (p >= threshold).astype(int)
    true_label = y_test[col].values.astype(int)

    df_tmp = pd.DataFrame({
        "model": "BNN",
        "patient_id": np.arange(len(p)),
        "label": col,
        "probability": p,
        "uncertainty": uncertainty,
        "uncertainty_pct": uncertainty * 100,
        "ambiguity": ambiguity,
        "threshold": threshold,
        "true_label": true_label,
        "prediction": prediction,
        "correct": prediction == true_label,
        "distance_from_threshold": np.abs(p - threshold),
    })
    df_tmp["error_type"] = np.select(
        [
            (df_tmp["true_label"] == 1) & (df_tmp["prediction"] == 1),
            (df_tmp["true_label"] == 0) & (df_tmp["prediction"] == 0),
            (df_tmp["true_label"] == 0) & (df_tmp["prediction"] == 1),
            (df_tmp["true_label"] == 1) & (df_tmp["prediction"] == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="NA",
    )
    patient_rows.append(df_tmp)

df_patient_level_bnn = pd.concat(patient_rows, ignore_index=True)

with open(PROJECT_DIR / "patient_level_bnn.pkl", "wb") as f:
    pickle.dump(df_patient_level_bnn, f)

print(f"Saved patient-level results to {PROJECT_DIR / 'patient_level_bnn.pkl'}")
df_patient_level_bnn.head()


In [24]:
import pickle

with open('model_final_bnn.pkl', 'wb') as f:
    pickle.dump(model_final, f)